# 08 — Long-Form Inference: Full-Game Video

Runs each detector+tracker combination on a **full, uncut game video**
(`data/processed/labelstudio/{GAME_PREFIX}.mp4`) and accumulates the Bayesian
minigame posterior over the entire video with ground-truth comparison.

Because there are no chunk boundaries, track IDs are continuous throughout —
no cross-chunk stitching is required. `merge_fragments()` is still applied
post-hoc to bridge genuine re-entry gaps within the video.

Ground truth is built by concatenating per-chunk `gt.txt` files with frame
and track-ID offsets (as produced by `01_prepare_dataset.ipynb`).

Prerequisites: `02_train_yolo.ipynb` · `03_train_rtdetr.ipynb` · `04_tracker_architecture_botsort.ipynb` · `04b_tracker_architecture_bytetrack.ipynb`

## 0 — Configuration

In [ ]:
from pathlib import Path
import json, cv2, yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.auto import tqdm
from ultralytics import YOLO

REPO_ROOT    = Path('/home1/hendersonj6179@cgu.edu')
RUNS_DIR     = (REPO_ROOT / 'runs').resolve()
MOT_SEQ_DIR  = REPO_ROOT / 'data/mot_dataset/sequences'
CHUNKS_DIR   = REPO_ROOT / 'data/processed/labelstudiochunks'
OUT_DIR      = (REPO_ROOT / 'runs/longform').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Select the full game to analyse ──────────────────────────────────────
# Set to the shared prefix of all chunks belonging to one game.
# All sequences whose name starts with this prefix will be used in order.
GAME_PREFIX = 'AY_G1'    # e.g. 'AY_G1', 'SerpentBoi_G2', 'Frozoha_G1', ...

# ── Detector weights ──────────────────────────────────────────────────────
WEIGHTS     = RUNS_DIR / 'yolo26' / 'final' / 'weights' / 'best.pt'
# WEIGHTS   = RUNS_DIR / 'rtdetr' / 'final' / 'weights' / 'best.pt'

# ── Tracker config: baseline arch config from notebook 04 ────────────────
# HP tuning (notebooks 05/05b) was abandoned — baseline outperforms tuned configs.
_ARCH_CFG_PATH = REPO_ROOT / 'configs' / 'kartector_botsort_arch.json'
if _ARCH_CFG_PATH.exists():
    import json as _jcfg08
    _arch_cfg08   = _jcfg08.loads(_ARCH_CFG_PATH.read_text())
    TRACKER_CFG   = str(REPO_ROOT / _arch_cfg08.get('tracker_yaml', 'configs/kartector_botsort_reentry.yaml'))
    CONF          = _arch_cfg08.get('conf', 0.25)
    IOU           = 0.70
    REID_OPTION   = _arch_cfg08.get('reid_option', 'B')
    REID_B_PARAMS = _arch_cfg08.get('reid_b_params', {})
else:
    TRACKER_CFG   = str(REPO_ROOT / 'configs' / 'kartector_botsort_reentry.yaml')
    CONF = 0.25; IOU = 0.70
    REID_OPTION   = 'B'
    REID_B_PARAMS = {}
if not Path(TRACKER_CFG).exists():
    TRACKER_CFG = 'botsort.yaml'

def _make_boxmot_tracker08():
    """Instantiate a fresh BotSort tracker using the baseline arch config."""
    import torch
    from boxmot.trackers.botsort.botsort import BotSort
    from boxmot.utils import WEIGHTS as _BW
    device = torch.device(
        'cuda' if torch.cuda.is_available() else
        'mps'  if torch.backends.mps.is_available() else 'cpu')
    return BotSort(
        reid_weights=_BW / 'osnet_x0_25_msmt17.pt',
        device=device, half=False,
        track_buffer=REID_B_PARAMS.get('track_buffer', 30),
        match_thresh=REID_B_PARAMS.get('match_thresh', 0.70),
        appearance_thresh=REID_B_PARAMS.get('appearance_thresh', 0.40),
        proximity_thresh=REID_B_PARAMS.get('proximity_thresh', 0.5),
        with_reid=True,
    )

with open(REPO_ROOT / 'data/splits/splits.json') as f:
    splits = json.load(f)
CLASSES   = splits['classes']
N_CLASSES = len(CLASSES)

# Discover all sequences for this game, in chunk order
all_seqs = sorted(MOT_SEQ_DIR.iterdir())
game_seqs = [s for s in all_seqs if s.name.startswith(GAME_PREFIX) and s.is_dir()]
print(f'Game prefix : {GAME_PREFIX}')
print(f'Chunks found: {len(game_seqs)}')
print(f'Weights     : {WEIGHTS}')
print(f'Tracker     : {TRACKER_CFG}')

## 1 — Minigame Model

`compute_posterior` is imported from `helpers.py`.

> **TODO:** Replace placeholder distributions with real drop-rate values once confirmed.

In [ ]:
from helpers import compute_posterior

MINIGAMES = [
    "Single Race",
    "Drag Race",
    "Destruction Derby",
]
PRIOR = np.ones(len(MINIGAMES)) / len(MINIGAMES)
LIKELIHOODS = np.array([
    [0.083,0.083,0.083,0.083,0.075,0.062,0.083,0.083,0.083],  # Single Race
    [0.082,0.082,0.082,0.082,0.074,0.082,0.082,0.082,0.074],  # Drag Race
    [0.09, 0.072,0.108,0.072,0.072,0.129,0.05, 0.072,0.072],  # Destruction Derby
])
LIKELIHOODS = LIKELIHOODS / LIKELIHOODS.sum(axis=1, keepdims=True)
assert LIKELIHOODS.shape == (len(MINIGAMES), N_CLASSES)
print('Minigame model ready.')

## 2 — Build Ground-Truth Long-Form Timeline

Reads per-chunk `gt.txt` files and concatenates them with frame and track-ID offsets
to produce a single continuous timeline.

GT format (MOT17): `frame, track_id, x, y, w, h, conf, class_id, visibility`
(1-indexed frames; class_id is 1-indexed in file, converted to 0-indexed here)

In [ ]:
def load_gt_chunk(seq_dir):
    gt_file = seq_dir / 'gt' / 'gt.txt'
    rows = []
    if not gt_file.exists():
        return pd.DataFrame(columns=['frame','track_id','class_id','x1','y1','x2','y2'])
    for line in gt_file.read_text().splitlines():
        parts = line.strip().split(',')
        if len(parts) < 8: continue
        frame_id = int(parts[0]) - 1          # 0-indexed
        track_id = int(parts[1])
        x, y, w, h = float(parts[2]), float(parts[3]), float(parts[4]), float(parts[5])
        class_id  = int(parts[7]) - 1         # MOT gt.txt is 1-based; convert to 0-based
        rows.append({'frame': frame_id, 'track_id': track_id, 'class_id': class_id,
                     'x1': x, 'y1': y, 'x2': x + w, 'y2': y + h})
    return pd.DataFrame(rows)


def get_seq_length(seq_dir):
    ini = seq_dir / 'seqinfo.ini'
    if ini.exists():
        for line in ini.read_text().splitlines():
            if line.startswith('seqLength'):
                return int(line.split('=')[1])
    # fallback: count images
    return len(list((seq_dir / 'img1').glob('*.jpg')))


gt_rows_all = []
frame_offset = 0
tid_offset   = 0
chunk_boundaries = []   # (chunk_name, global_start_frame, global_end_frame)
TID_STRIDE = 100_000    # large enough to prevent any cross-chunk ID collision

for seq in tqdm(game_seqs, desc='Loading GT'):
    chunk_df = load_gt_chunk(seq)
    seq_len  = get_seq_length(seq)
    if not chunk_df.empty:
        chunk_df['frame']    += frame_offset
        chunk_df['track_id'] += tid_offset
        gt_rows_all.append(chunk_df)
    chunk_boundaries.append((seq.name, frame_offset, frame_offset + seq_len - 1))
    frame_offset += seq_len
    tid_offset   += TID_STRIDE

gt_df = pd.concat(gt_rows_all, ignore_index=True) if gt_rows_all else pd.DataFrame(
    columns=['frame','track_id','class_id','x1','y1','x2','y2'])
total_frames = frame_offset

# Apply merge_fragments to GT to stitch tracks broken at chunk boundaries.
# Within each chunk tracks are already correct; only cross-boundary resets need repair.
gt_df_raw = gt_df.copy()
gt_df      = merge_fragments(gt_df, max_gap=60, max_dist_pct=35.0)
_merged_gt = gt_df_raw['track_id'].nunique() - gt_df['track_id'].nunique()
print(f'GT track IDs before merge : {gt_df_raw["track_id"].nunique()}')
print(f'GT track IDs after  merge : {gt_df["track_id"].nunique()}  ({_merged_gt} stitched)')


print(f'Total frames (GT timeline): {total_frames}')
print(f'GT detections             : {len(gt_df)}')
print(f'GT unique track IDs       : {gt_df["track_id"].nunique()}')
print(f'Chunks with annotations   : {sum(1 for c in game_seqs if (c/"gt"/"gt.txt").exists())} / {len(game_seqs)}')

## 3 — Locate the Full-Game Video

The original full-game videos live at `data/processed/labelstudio/{GAME_PREFIX}.mp4`.
A frame-count check is printed to verify alignment with the GT timeline.

In [ ]:
# Full-game video lives at data/processed/{GAME_PREFIX}.mp4
FULL_VIDEO_DIR = REPO_ROOT / 'data' / 'processed' / 'labelstudio'
FULL_VIDEO     = FULL_VIDEO_DIR / f'{GAME_PREFIX}.mp4'

if not FULL_VIDEO.exists():
    raise FileNotFoundError(
        f'Full-game video not found: {FULL_VIDEO}\n'
        f'Expected data/processed/{GAME_PREFIX}.mp4')

_cap_check = cv2.VideoCapture(str(FULL_VIDEO))
_full_frames = int(_cap_check.get(cv2.CAP_PROP_FRAME_COUNT))
_full_fps    = _cap_check.get(cv2.CAP_PROP_FPS)
_cap_check.release()
print(f'Full-game video : {FULL_VIDEO.name}')
print(f'  Frames        : {_full_frames} @ {_full_fps:.1f} fps')
print(f'  GT timeline   : {total_frames} frames')
if abs(_full_frames - total_frames) > 10:
    print(f'  WARNING: frame count mismatch ({_full_frames} vs {total_frames}). '
          f'GT/tracker posteriors may be misaligned.')
else:
    print('  Frame counts match — ready to track.')


## 4 — Run All 4 Trackers on the Full-Game Video

Each tracker processes the full video in a single call — no resets, no stitching.
`merge_fragments()` is applied post-hoc to bridge genuine re-entry gaps.

In [ ]:
from helpers import merge_fragments

## 5 — Accumulate Posteriors

In [ ]:
from helpers import accumulate_longform

## 6 — Side-by-Side Posterior Evolution Plot

One GT vs tracker pair per run. Chunk boundaries are marked for reference even
though the tracker saw a continuous video.

In [ ]:
# ── Plot: posterior evolution for each run vs GT ─────────────────────────
colors = plt.cm.Set1.colors
x = np.arange(total_frames)

for lbl, res in all_run_results.items():
    fig, axes = plt.subplots(2, 1, figsize=(20, 8), sharey=True, sharex=True)
    for ax, posts, title in zip(axes,
                                 [gt_posts,       res['posts']],
                                 ['Ground Truth', lbl]):
        for i, mg in enumerate(MINIGAMES):
            ax.plot(x, posts[:, i], label=mg, lw=1.5, color=colors[i % len(colors)])
        for _, start, _ in chunk_boundaries:
            ax.axvline(start, color='gray', lw=0.5, alpha=0.4)
        ax.set_ylabel('P(minigame)'); ax.set_ylim(0, 1.05)
        ax.set_title(f'{title} — {GAME_PREFIX}', fontweight='bold')
        ax.legend(fontsize=8, loc='upper right'); ax.grid(alpha=0.2)
    axes[1].set_xlabel('Frame (global)')
    plt.suptitle(f'{lbl} vs GT — {GAME_PREFIX}', fontweight='bold', y=1.01)
    plt.tight_layout()
    _slug = lbl.replace(' ', '_').replace('+', 'x')
    _out  = OUT_DIR / f'{GAME_PREFIX}_{_slug}_posterior_evolution.png'
    plt.savefig(_out, dpi=150, bbox_inches='tight'); plt.show()
    print(f'Saved: {_out}')


## 7 — Overlay Comparison & Divergence

In [ ]:
# ── Overlay + divergence for each run ────────────────────────────────────
for lbl, res in all_run_results.items():
    trk_posts = res['posts']
    fig, axes = plt.subplots(2, 1, figsize=(20, 8), sharex=True,
                             gridspec_kw={'height_ratios': [3, 1]})
    for i, mg in enumerate(MINIGAMES):
        c = colors[i % len(colors)]
        axes[0].plot(x, gt_posts[:, i],  color=c, lw=1.5,       label=f'{mg} (GT)')
        axes[0].plot(x, trk_posts[:, i], color=c, lw=1.5, ls='--', alpha=0.7, label=f'{mg} (Trk)')
    for _, start, _ in chunk_boundaries:
        axes[0].axvline(start, color='gray', lw=0.5, alpha=0.4)
    axes[0].set_ylabel('P(minigame)'); axes[0].set_ylim(0, 1.05)
    axes[0].set_title(f'GT vs {lbl} — {GAME_PREFIX}', fontweight='bold')
    axes[0].legend(fontsize=7, ncol=len(MINIGAMES)*2, loc='upper right')
    axes[0].grid(alpha=0.2)
    divergence = np.abs(gt_posts - trk_posts).sum(axis=1) / 2
    axes[1].fill_between(x, divergence, alpha=0.4, color='crimson')
    axes[1].plot(x, divergence, color='crimson', lw=1)
    axes[1].set_ylabel('L1 divergence'); axes[1].set_ylim(0, 1)
    axes[1].set_xlabel('Frame (global)')
    axes[1].set_title('Posterior Divergence (0=identical, 1=opposite)', fontweight='bold')
    for _, start, _ in chunk_boundaries:
        axes[1].axvline(start, color='gray', lw=0.5, alpha=0.4)
    axes[1].grid(alpha=0.2)
    plt.tight_layout()
    _slug = lbl.replace(' ', '_').replace('+', 'x')
    _out  = OUT_DIR / f'{GAME_PREFIX}_{_slug}_divergence.png'
    plt.savefig(_out, dpi=150, bbox_inches='tight'); plt.show()
    res['mean_divergence'] = float(divergence.mean())
    res['max_divergence']  = float(divergence.max())
    res['max_div_frame']   = int(divergence.argmax())
    print(f'{lbl}: mean_div={divergence.mean():.3f}  max_div={divergence.max():.3f}')


## 8 — Stat Count Comparison & Summary Table

Per-run stat count tables and a summary table. Results saved to
`{GAME_PREFIX}_longform_summary.csv`.

In [ ]:
# ── Per-run stat count table ──────────────────────────────────────────────
for lbl, res in all_run_results.items():
    _df = pd.DataFrame({
        'GT counts':      gt_counts,
        'Tracker counts': res['counts'],
        'Difference':     res['counts'] - gt_counts,
        '% Error':        np.where(gt_counts > 0,
                                   (res['counts'] - gt_counts) / gt_counts * 100, np.nan),
    }, index=CLASSES)
    print(f'\n── {lbl} — Stat Count Comparison ──')
    display(_df.style.format({'% Error': '{:.1f}%'})
                     .background_gradient(subset=['Difference'], cmap='RdYlGn_r'))

# ── Summary table ─────────────────────────────────────────────────────────
summary_rows = []
for lbl, res in all_run_results.items():
    final_post_trk = res['posts'][-1]
    final_post_gt  = gt_posts[-1]
    summary_rows.append({
        'Run':             lbl,
        'Elapsed (s)':     round(res['elapsed'], 1),
        'Elapsed (min)':   round(res['elapsed'] / 60, 2),
        'Mean Divergence': round(res.get('mean_divergence', float('nan')), 4),
        'Max Divergence':  round(res.get('max_divergence',  float('nan')), 4),
        'Unique Tracks':   int(res['trk_df']['track_id'].nunique()),
        'Total Counts':    int(res['counts'].sum()),
        'GT Counts':       int(gt_counts.sum()),
        'Count % Error':   round(float(np.where(gt_counts.sum() > 0,
                               (res['counts'].sum() - gt_counts.sum()) / gt_counts.sum() * 100,
                               float('nan'))), 2),
    })

df_summary = pd.DataFrame(summary_rows).set_index('Run')
print('\n' + '='*60)
print(f'SUMMARY — {GAME_PREFIX}')
print('='*60)
display(df_summary.style
    .format({'Elapsed (s)': '{:.1f}', 'Elapsed (min)': '{:.2f}',
             'Mean Divergence': '{:.4f}', 'Max Divergence': '{:.4f}',
             'Count % Error': '{:+.1f}%'})
    .background_gradient(subset=['Mean Divergence'], cmap='YlOrRd')
    .background_gradient(subset=['Elapsed (s)'],     cmap='Blues')
    .background_gradient(subset=['Count % Error'],   cmap='RdYlGn_r'))

df_summary.to_csv(OUT_DIR / f'{GAME_PREFIX}_fullvideo_run_summary.csv')
print(f'\nSaved: {OUT_DIR}/{GAME_PREFIX}_run_summary.csv')
